In [4]:
import cv2
import os
import json
import numpy as np
import albumentations as A
from datetime import datetime

# ===== CONFIG =====
INPUT_IMG_DIR = "labeled_example_data"
INPUT_ANN_FILE = os.path.join(INPUT_IMG_DIR, "labels.json")

# Create timestamped output folder
timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
BASE_OUTPUT_DIR = f"synthetic_{timestamp}"
OUTPUT_IMG_DIR = os.path.join(BASE_OUTPUT_DIR, "images")
OUTPUT_ANN_FILE = os.path.join(BASE_OUTPUT_DIR, "annotations.json")
OUTPUT_HM_DIR = os.path.join(BASE_OUTPUT_DIR, "heatmaps")

NUM_VARIANTS = 5
HEATMAP_SIZE = (128, 128)  # match your model input
SIGMA = 2.0

os.makedirs(OUTPUT_IMG_DIR, exist_ok=True)
os.makedirs(OUTPUT_HM_DIR, exist_ok=True)

# ===== AUGMENTATION PIPELINE =====
transform = A.Compose([
    A.RandomBrightnessContrast(p=0.5),
    A.HueSaturationValue(p=0.5),
    A.MotionBlur(blur_limit=5, p=0.3),
    A.GaussNoise(var_limit=(10.0, 50.0), p=0.3),
    A.Affine(scale=(0.9, 1.1), translate_percent=(0.05, 0.1),
             rotate=(-15, 15), shear=(-5, 5), p=0.7),
    A.RandomShadow(p=0.3),
    A.RandomFog(p=0.2)
], keypoint_params=A.KeypointParams(format='xy', remove_invisible=False))

# ===== HEATMAP GENERATOR =====
def generate_heatmap(size, center, sigma):
    hm = np.zeros(size, dtype=np.float32)
    x, y = int(center[0]), int(center[1])
    if 0 <= x < size[1] and 0 <= y < size[0]:
        xx, yy = np.meshgrid(np.arange(size[1]), np.arange(size[0]))
        hm = np.exp(-((xx - x) ** 2 + (yy - y) ** 2) / (2 * sigma ** 2))
    return hm

# ===== LOAD ORIGINAL ANNOTATIONS =====
with open(INPUT_ANN_FILE, 'r') as f:
    annotations = json.load(f)

synthetic_annotations = []

# ===== GENERATE SYNTHETIC DATA =====
for ann in annotations:
    if ann.get("is_present", 1.0) != 1.0:
        continue  # skip absent rims if needed

    img_path = os.path.join(INPUT_IMG_DIR, ann["filename"])
    image = cv2.imread(img_path)
    if image is None:
        print(f"Warning: Could not read {img_path}, skipping.")
        continue

    h, w = image.shape[:2]
    # Convert normalized coords to pixel coords
    px = ann["center"][0] * w
    py = ann["center"][1] * h
    keypoints = [(px, py)]

    for i in range(NUM_VARIANTS):
        augmented = transform(image=image, keypoints=keypoints)
        aug_img = augmented["image"]
        aug_kps = augmented["keypoints"]

        out_name = f"synthetic_{i}_{ann['filename']}"
        cv2.imwrite(os.path.join(OUTPUT_IMG_DIR, out_name), aug_img)

        # Resize keypoint to heatmap size
        scale_x = HEATMAP_SIZE[1] / aug_img.shape[1]
        scale_y = HEATMAP_SIZE[0] / aug_img.shape[0]
        kp_scaled = (aug_kps[0][0] * scale_x, aug_kps[0][1] * scale_y)

        # Generate and save heatmap
        hm = generate_heatmap(HEATMAP_SIZE, kp_scaled, SIGMA)
        hm_path = os.path.join(OUTPUT_HM_DIR, out_name.replace(".png", "_hm.npy"))
        np.save(hm_path, hm)

        synthetic_annotations.append({
            "filename": out_name,
            "center": [
                float(aug_kps[0][0] / aug_img.shape[1]),
                float(aug_kps[0][1] / aug_img.shape[0])
            ],  # normalized coords
            "heatmap_file": hm_path,
            "is_present": 1.0
        })

# ===== SAVE SYNTHETIC ANNOTATIONS =====
with open(OUTPUT_ANN_FILE, 'w') as f:
    json.dump(synthetic_annotations, f, indent=2)

print(f"Generated {len(synthetic_annotations)} synthetic samples "
      f"({NUM_VARIANTS} per original present rim) in folder: {BASE_OUTPUT_DIR}")

/var/folders/8m/p7s3fq9j1y55c3pc57sg7z2r0000gn/T/ipykernel_92413/2295461363.py:31: UserWarning: Argument(s) 'var_limit' are not valid for transform GaussNoise
  A.GaussNoise(var_limit=(10.0, 50.0), p=0.3),


Generated 5000 synthetic samples (5 per original present rim) in folder: synthetic_20250911_210245


Center Labeling Tool (Generates labels.json)